In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots

include("functions.jl")
Random.seed!(2025)
const bb = 27 
const aa = 38
const N  = 1462439
const I0 = 1
const S0 = 1316195
const n_iter = 1_000_000

Istar_obs = [3, 7, 10, 4, 14, 35, 92, 98, 216, 374, 434, 417, 447, 275, 151, 83, 67, 48, 37, 29, 42, 46, 53, 73, 87, 111, 123, 124, 99, 135, 106, 74, 39, 13]

tau = length(Istar_obs)

model_tag_sym = :sliding
KMAX_UPPER = 30  
KMAX_fixed = 29

# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 3

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER,
            k_max_fixed = KMAX_fixed)
    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [sliding_model] Fitting chain 3 (tau=34)
[ Info: [sliding] iter 1000/1000000 elapsed=3.9s, rate=0.099, mean=[1.539, 0.00053, 2.047], std=[0.3347, 0.000810, 0.2436] [ADAPT]
[ Info: [sliding] iter 2000/1000000 elapsed=6.8s, rate=0.058, mean=[1.952, 0.00031, 2.081], std=[0.4496, 0.000609, 0.1756] [ADAPT]
[ Info: [sliding] iter 3000/1000000 elapsed=9.0s, rate=0.040, mean=[2.131, 0.00024, 2.099], std=[0.4315, 0.000509, 0.1452] [ADAPT]
[ Info: [sliding] iter 4000/1000000 elapsed=11.2s, rate=0.032, mean=[2.220, 0.00020, 2.107], std=[0.3978, 0.000447, 0.1263] [ADAPT]
[ Info: [sliding] iter 5000/1000000 elapsed=13.4s, rate=0.028, mean=[2.264, 0.00018, 2.113], std=[0.3654, 0.000404, 0.1135] [ADAPT]
[ Info: [sliding] iter 6000/1000000 elapsed=15.6s, rate=0.025, mean=[2.261, 0.00016, 2.125], std=[0.3338, 0.000371, 0.1068] [ADAPT]
[ Info: [sliding] iter 7000/1000000 elapsed=17.8s, rate=0.024, mean=[2.260, 0.00016, 2.137], std=[0.3095, 0.000346, 0.1024] [ADAPT]
[ Info: [sliding] iter 8000/10